In [1]:
import os
os.environ['KERAS_BACKEND'] = 'torch'  # You can change to 'jax' or 'torch' if preferred

import keras

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.config.backend()}")

C:\Users\Patrik\anaconda3\envs\pyTorch\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Keras version: 3.14.0
Keras backend: torch


In [2]:
#pip install transformers
#pip install librosa
#pip install ipywidgets
#pip install PySoundFile
#conda install -c conda-forge libsndfile
#pip install sentencepiece
#pip install sacremoses
#!pip install torchcodec

In [3]:
import torch
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset

# Background

Background chapter written with information sourced from:  
https://medium.com/analytics-vidhya/understanding-the-mel-spectrogram-fca2afa2ce53

The use of audio files as pure raw data is unviable, for the same reason neural networks struggle with complex coloured images in image classification.  
In the process of vectorising the input data a lot of locational contextual information is lost. Due to audio being linear with time this applies even more so.  
Normally this wouldn't pose an issue as arrays would do well in representating any linearity, but the information about frequencies and amplitudes of the sound will get lost in this translation.  

In an answer to this the captured audio will first, through applying **Fourier transform**, be split up from its form of multiple comprised soundwaves into their individual frequencies and amplitudes.  
This when done over multiple segments of an audio recording through a process called **short-time Fourier transform** creates a spectogram, in which the **x-axis** represents time and **y-axis** a logarithimic scale for the sound. In doing so in the spectogram there will appear spots in the image where a lot of frequencies stack on top of each other. These are the spots that get highlighted in brighter colour in an image representation.  

One final step from here is to convert the resulting spectogram into a mel spectogram by following mel scale which takes into account the way in which humans perceive sound. 
This creates an image from which a neural network can easily recognize and categorise patterns in audio.

In [4]:
audio_file = load_dataset("shunyalabs/finnish-speech-dataset", "default", split='train')

In [5]:
sample = audio_file[0]

RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.11.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call 

In [ ]:
# Load audio file
y, sr = librosa.load(sample, sr=16000)

# Display audio for playback
print("Play the audio:")
display(Audio(y, rate=sr))

# Print audio statistics
duration = librosa.get_duration(y=y, sr=sr)
print(f"\nAudio Statistics:")
print(f"Duration: {duration:.2f} seconds")
print(f"Sample rate: {sr} Hz")

In [ ]:
# Initialize the model
model_size = "tiny"  # Options: tiny, base, small, medium, large
device = "cuda" if torch.cuda.is_available() else "cpu"
device = "mps" if torch.mps.is_available() else "cpu"
print(f"Using device: {device}")

# Load model and processor
print(f"Loading Whisper {model_size} model...")
model_name = f"openai/whisper-{model_size}"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name).to(device)

# Set the language to Finnish
forced_decoder_ids = processor.get_decoder_prompt_ids(language="fi", task="transcribe")

# Process with WhisperProcessor
print("Processing audio...")
input_features = processor(
    y, 
    sampling_rate=sr, 
    return_tensors="pt"
).input_features.to(device)

# Generate the transcription
print("Generating transcription...")
with torch.no_grad():
    generated_ids = model.generate(
        input_features,
        language="fi",
        task="transcribe"
    )

# Decode the generated IDs
transcription = processor.batch_decode(
    generated_ids, 
    skip_special_tokens=True
)[0]

print("\nFINNISH TRANSCRIPTION:")
print("="*50)
print(transcription)